## Example 1: Basic HTTP/2 frame creation

In [ ]:
from hyperframe.frame import Frame, DataFrame

# Example 1: Basic HTTP/2 frame creation
print("Example 1: Basic HTTP/2 frame creation")

# Create a basic HTTP/2 DATA frame
data_frame = DataFrame(stream_id=1)
data_frame.data = b"Hello, HTTP/2!"

print(f"Data frame created: {type(data_frame).__name__}")
print(f"Stream ID: {data_frame.stream_id}")
print(f"Frame type: {data_frame.type}")
print(f"Data length: {len(data_frame.data)}")
print(f"Data content: {data_frame.data}")

# Serialize the frame
serialized = data_frame.serialize()
print(f"Serialized frame length: {len(serialized)} bytes")
print(f"Serialized data (hex): {serialized[:10].hex()}...")

# Show frame flags
print(f"Frame flags: {data_frame.flags}")

# 9-byte header (length 14, type 0 = DATA, no flags, stream 1) followed by the payload
assert data_frame.type == 0
assert len(serialized) == 9 + len(data_frame.data)
assert serialized[:9] == bytes.fromhex("00000e000000000001")
parsed, length = Frame.parse_frame_header(serialized[:9])
parsed.parse_body(memoryview(serialized[9:9 + length]))
assert isinstance(parsed, DataFrame) and parsed.data == b"Hello, HTTP/2!" and parsed.stream_id == 1

## Example 2: HTTP/2 HEADERS frame handling

In [ ]:
from hyperframe.frame import HeadersFrame

# Example 2: HTTP/2 HEADERS frame handling
print("Example 2: HTTP/2 HEADERS frame handling")

# Create a HEADERS frame
headers_frame = HeadersFrame(stream_id=3)
headers_frame.data = b":method GET\r\n:path /api/users\r\n:authority example.com\r\n"

print(f"Headers frame created: {type(headers_frame).__name__}")
print(f"Stream ID: {headers_frame.stream_id}")
print(f"Frame type: {headers_frame.type}")
print(f"Headers data: {headers_frame.data}")

# Set frame flags
headers_frame.flags.add('END_HEADERS')
headers_frame.flags.add('END_STREAM')

print(f"Frame flags: {list(headers_frame.flags)}")
print(f"END_HEADERS flag set: {'END_HEADERS' in headers_frame.flags}")
print(f"END_STREAM flag set: {'END_STREAM' in headers_frame.flags}")

# Serialize the frame
serialized = headers_frame.serialize()
print(f"Serialized headers frame: {len(serialized)} bytes")

# HEADERS is frame type 1; END_STREAM (0x1) | END_HEADERS (0x4) = flags byte 0x05
assert headers_frame.type == 1
assert serialized[3] == 1 and serialized[4] == 0x05
from hyperframe.frame import Frame
parsed, length = Frame.parse_frame_header(serialized[:9])
parsed.parse_body(memoryview(serialized[9:9 + length]))
assert isinstance(parsed, HeadersFrame) and parsed.stream_id == 3
assert set(parsed.flags) == {"END_HEADERS", "END_STREAM"} and parsed.data == headers_frame.data

## Example 3: HTTP/2 SETTINGS frame configuration

In [ ]:
from hyperframe.frame import SettingsFrame

# Example 3: HTTP/2 SETTINGS frame configuration
print("Example 3: HTTP/2 SETTINGS frame configuration")

# Create a SETTINGS frame
settings_frame = SettingsFrame(stream_id=0)

# Add HTTP/2 settings
settings = {
    SettingsFrame.HEADER_TABLE_SIZE: 4096,
    SettingsFrame.ENABLE_PUSH: 1,
    SettingsFrame.MAX_CONCURRENT_STREAMS: 100,
    SettingsFrame.INITIAL_WINDOW_SIZE: 65535
}

settings_frame.settings = settings

print(f"Settings frame created: {type(settings_frame).__name__}")
print(f"Stream ID: {settings_frame.stream_id}")
print(f"Frame type: {settings_frame.type}")
print(f"Number of settings: {len(settings_frame.settings)}")

print("Settings configured:")
for setting_id, value in settings_frame.settings.items():
    print(f"  Setting {setting_id}: {value}")

# Serialize settings frame
serialized = settings_frame.serialize()
print(f"Serialized settings frame: {len(serialized)} bytes")

# SETTINGS (type 4) carries 6 bytes per setting
assert settings_frame.type == 4 and len(serialized) == 9 + 6 * len(settings)
from hyperframe.frame import Frame
parsed, length = Frame.parse_frame_header(serialized[:9])
parsed.parse_body(memoryview(serialized[9:9 + length]))
assert parsed.settings == settings

# SETTINGS frames belong to the connection, so a non-zero stream id is rejected
from hyperframe.exceptions import InvalidDataError
try:
    SettingsFrame(stream_id=1)
except InvalidDataError as e:
    print(f"Rejected stream 1: {e}")
else:
    raise AssertionError("SettingsFrame accepted a non-zero stream id")

## Example 4: Flow control and ping frames

In [ ]:
from hyperframe.frame import WindowUpdateFrame, PingFrame

# Example 4: Flow control and ping frames
print("Example 4: Flow control and ping frames")

# Create a WINDOW_UPDATE frame
window_frame = WindowUpdateFrame(stream_id=1)
window_frame.window_increment = 32768

print(f"Window update frame: {type(window_frame).__name__}")
print(f"Stream ID: {window_frame.stream_id}")
print(f"Window size increment: {window_frame.window_increment}")
print(f"Frame type: {window_frame.type}")

# Create a PING frame
ping_frame = PingFrame(stream_id=0)
ping_frame.opaque_data = b"12345678"  # 8-byte ping data

print(f"Ping frame: {type(ping_frame).__name__}")
print(f"Stream ID: {ping_frame.stream_id}")
print(f"Ping data: {ping_frame.opaque_data}")
print(f"Ping data length: {len(ping_frame.opaque_data)}")

# Serialize both frames
window_serialized = window_frame.serialize()
ping_serialized = ping_frame.serialize()

print(f"Window frame serialized: {len(window_serialized)} bytes")
print(f"Ping frame serialized: {len(ping_serialized)} bytes")

assert (window_frame.type, ping_frame.type) == (8, 6)
assert len(window_serialized) == 9 + 4 and len(ping_serialized) == 9 + 8
from hyperframe.frame import Frame
parsed, length = Frame.parse_frame_header(window_serialized[:9])
parsed.parse_body(memoryview(window_serialized[9:9 + length]))
assert isinstance(parsed, WindowUpdateFrame) and parsed.window_increment == 32768
parsed, length = Frame.parse_frame_header(ping_serialized[:9])
parsed.parse_body(memoryview(ping_serialized[9:9 + length]))
assert isinstance(parsed, PingFrame) and parsed.opaque_data == b"12345678"

## Example 5: Frame parsing and deserialization

In [ ]:
from hyperframe.frame import Frame

# Example 5: Frame parsing and deserialization
print("Example 5: Frame parsing and deserialization")

# Simulate receiving frame data (9-byte frame header + payload)
# This represents a minimal DATA frame
frame_data = bytearray([
    0x00, 0x00, 0x0C,  # Length: 12 bytes
    0x00,              # Type: DATA frame (0)
    0x01,              # Flags: END_STREAM
    0x00, 0x00, 0x00, 0x01,  # Stream ID: 1
    # Payload: "Hello HTTP/2"
    0x48, 0x65, 0x6C, 0x6C, 0x6F, 0x20, 0x48, 0x54, 0x54, 0x50, 0x2F, 0x32
])

print(f"Raw frame data length: {len(frame_data)} bytes")
print(f"Frame data (hex): {frame_data.hex()}")

# Parse the header, then the body; parse_frame_header returns the frame and the payload length
parsed_frame, length = Frame.parse_frame_header(frame_data[:9])
parsed_frame.parse_body(memoryview(frame_data[9:9 + length]))
print(f"Frame parsed: {type(parsed_frame).__name__}")
print(f"Stream ID: {parsed_frame.stream_id}")
print(f"Flags: {list(parsed_frame.flags)}")
print(f"Payload: {parsed_frame.data}")

from hyperframe.frame import DataFrame
assert isinstance(parsed_frame, DataFrame) and parsed_frame.type == 0
assert length == 12 and parsed_frame.stream_id == 1
assert set(parsed_frame.flags) == {"END_STREAM"}
assert parsed_frame.data == b"Hello HTTP/2"